In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import warnings
import pandas as pd
import yfinance as yf
import time
from helpers import filter_daily_volume, drop_nan_days, get_all_us_tickers, get_data, filter_tickers
import numpy as np
import matplotlib.pyplot as plt
from TradingSim import TradingSim

In [3]:
tmp = pd.read_parquet("AllUSData-20100101-20260303.parquet")

In [4]:
full_df = tmp.copy()
qqq_spy = yf.download(["QQQ", "SPY"], start="2010-01-01", end="2026-03-03")
full_df = pd.concat([full_df, qqq_spy], axis=1)
training_df = full_df.loc["2010-01-01": "2020-12-31"]
testing_df = full_df.loc["2019-01-01":]  # Takes a year before so it will have time to prepare data

C:\Users\sasso\AppData\Local\Temp\ipykernel_28844\2819213082.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  qqq_spy = yf.download(["QQQ", "SPY"], start="2010-01-01", end="2026-03-03")
[*********************100%***********************]  2 of 2 completed


In [7]:
def calc_window_drawdown_numba(prices):
    max_price = prices[0]
    max_dd = 0.0
    
    for i in range(len(prices)):
        if prices[i] > max_price:
            max_price = prices[i]
            
        # SAFETY CHECK: If max_price is zero or negative, skip the drawdown calculation to prevent crashing
        if max_price <= 0:
            continue
            
        dd = (prices[i] - max_price) / max_price
        
        if dd < max_dd:
            max_dd = dd
            
    return max_dd * 100

def load_scores(df, year=252):
    close = df["Close"]
    # Filters
    vol_dol = (close * df["Volume"]).rolling(window="365D").median()
    vol_dol.columns = pd.MultiIndex.from_product([["vol dollar"], vol_dol.columns])
    nan_rolling = close.isna().rolling(window="365D").mean()
    nan_rolling.columns = pd.MultiIndex.from_product([["nan days %"], nan_rolling.columns])
    yearly_ret = (close.pct_change(periods=year) * 100)
    filters_mask = (vol_dol.values < 2000000) | (nan_rolling.values > 0.02)
    base_daily_ret = close.pct_change()
    volatility = (base_daily_ret.rolling(window=year).std() * np.sqrt(year) * 100).mask(filters_mask)
    
    steady_score = (yearly_ret / volatility).mask(filters_mask)
    positive_days = ((base_daily_ret > 0).rolling(window="365D").mean() * 100).mask(filters_mask)
    max_drawdown = close.rolling(window=year, min_periods=1).apply(
        calc_window_drawdown_numba, 
        engine='numba', 
        raw=True
    ).mask(filters_mask)
    # rolling_max = close.rolling(window=year, min_periods=1).max()
    # daily_drawdown = (close - rolling_max) / rolling_max
    # max_drawdown = (daily_drawdown.rolling(window=year).min() * 100).mask(filters_mask)
    sma_50 = close.rolling(window=50).mean()
    dist_sma_50 = ((close / sma_50 - 1) * 100).mask(filters_mask)
    median_vol_30d = df["Volume"].rolling(window=30).median()
    vol_surge_ratio = (df["Volume"] / median_vol_30d).mask(filters_mask)
    daily_range = df["High"] - df["Low"]
    atr_14 = daily_range.rolling(window=14).mean()
    atr_pct = ((atr_14 / close) * 100).mask(filters_mask)
    # cum_max = np.maximum.accumulate(close)
    # drawdowns = (close - cum_max) / cum_max

    # Ranking
    rank_total_ret = yearly_ret.rank(axis=1, pct=True) * 100
    rank_steady = steady_score.rank(axis=1, pct=True) * 100
    rank_positive = positive_days.rank(axis=1, pct=True) * 100
    rank_volatility = volatility.rank(axis=1, pct=True, ascending=False) * 100
    rank_drawdown = max_drawdown.rank(axis=1, pct=True) * 100
    rank_sma = dist_sma_50.rank(axis=1, pct=True) * 100
    rank_vol_surge = vol_surge_ratio.rank(axis=1, pct=True) * 100
    rank_atr = atr_pct.rank(axis=1, pct=True, ascending=False) * 100

    # Weights
    weights = {
        "TotalReturn": 4,
        "Volatility": 3,
        "SteadyScore": 9,
        "PositiveDays": 3,
        "MaxDrawdown": 8,
        "DistFromSMA50": 2,
        "VolumeSurgeRatio": 1,
        "ATR": 5
    }

    final_score_raw = (
        (rank_total_ret * weights["TotalReturn"]) +
        (rank_volatility * weights["Volatility"]) +
        (rank_steady * weights["SteadyScore"]) +
        (rank_positive * weights["PositiveDays"]) +
        (rank_drawdown * weights["MaxDrawdown"]) +
        (rank_sma * weights["DistFromSMA50"]) +
        (rank_vol_surge * weights["VolumeSurgeRatio"]) +
        (rank_atr * weights["ATR"])
    )

    low_score = (rank_volatility.values < 90) | (yearly_ret.values < 80)

    # Rank the final score and calculate integer rank
    final_score_pct = (final_score_raw.rank(axis=1, pct=True) * 100).mask(low_score)
    final_rank = final_score_pct.rank(axis=1, ascending=False, method="min")

    # 6. Apply MultiIndex to the metrics you want to keep
    rank_steady.columns = pd.MultiIndex.from_product([["SteadyScore"], rank_steady.columns])
    yearly_ret.columns = pd.MultiIndex.from_product([["Yearly Ret %"], yearly_ret.columns])
    rank_total_ret.columns = pd.MultiIndex.from_product([["Yearly Ret"], rank_total_ret.columns])
    rank_volatility.columns = pd.MultiIndex.from_product([["Volatility"], rank_volatility.columns])
    positive_days.columns = pd.MultiIndex.from_product([["PositiveDays %"], positive_days.columns])
    rank_positive.columns = pd.MultiIndex.from_product([["PositiveDays"], rank_positive.columns])
    rank_drawdown.columns = pd.MultiIndex.from_product([["MaxDrawdown"], rank_drawdown.columns])
    rank_sma.columns = pd.MultiIndex.from_product([["DistFromSMA50"], rank_sma.columns])
    rank_vol_surge.columns = pd.MultiIndex.from_product([["VolumeSurgeRatio"], rank_vol_surge.columns])
    rank_atr.columns = pd.MultiIndex.from_product([["ATR"], rank_atr.columns])
    final_score_pct.columns = pd.MultiIndex.from_product([["FinalScore"], final_score_pct.columns])
    final_rank.columns = pd.MultiIndex.from_product([["Rank"], final_rank.columns])

    # 7. Concatenate everything together
    df = pd.concat([
        df, rank_steady, yearly_ret, rank_total_ret, rank_volatility, positive_days, rank_positive,
        rank_drawdown, rank_sma, rank_vol_surge, rank_atr, final_score_pct, final_rank, nan_rolling, vol_dol
    ], axis=1)
    return df
    
def get_scores(df_loc):
    scores_df = pd.DataFrame(index=df_loc["Rank"].index)
    scores_df["TotalReturn %"] = df_loc["Yearly Ret %"]
    scores_df["TotalReturn"] = df_loc["Yearly Ret"]
    scores_df["Volatility"] = df_loc["Volatility"]
    scores_df["SteadyScore"] = df_loc["SteadyScore"]
    scores_df["PositiveDays"] = df_loc["PositiveDays"]
    scores_df["PositiveDays %"] = df_loc["PositiveDays %"]
    scores_df["MaxDrawdown"] = df_loc["MaxDrawdown"]
    scores_df["DistFromSMA50"] = df_loc["DistFromSMA50"]
    scores_df["VolumeSurgeRatio"] = df_loc["VolumeSurgeRatio"]
    scores_df["ATR"] = df_loc["ATR"]
    scores_df["FinalScore"] = df_loc["FinalScore"]
    scores_df["Rank"] = df_loc["Rank"]
    return scores_df.dropna()

def get_tops(df):
    long_ranks = df["Rank"].stack().reset_index()
    long_ranks.columns = ["Date", "Ticker", "Rank"]
    long_ranks = long_ranks.sort_values(by=["Date", "Rank"])

    top_long = long_ranks.groupby("Date").head(5).copy()
    top_long["Position"] = top_long.groupby("Date").cumcount() + 1

    top_tickers = top_long.pivot(index="Date", columns="Position", values="Ticker")
    return top_tickers

In [ ]:
training_score = load_scores(training_df)
top = get_tops(training_score.drop(["QQQ", "SPY"], axis=1, level=1, errors='ignore'))
top

C:\Users\sasso\AppData\Local\Temp\ipykernel_28844\4258049907.py:27: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  yearly_ret = (close.pct_change(periods=year) * 100)
C:\Users\sasso\AppData\Local\Temp\ipykernel_28844\4258049907.py:29: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  base_daily_ret = close.pct_change()


Position,1,2,3,4,5
Date,,,,,
2011-06-30,OKE,NaN,NaN,NaN,NaN
2011-07-01,OKE,NaN,NaN,NaN,NaN
2011-07-05,OKE,NaN,NaN,NaN,NaN
2011-07-15,TDG,NaN,NaN,NaN,NaN
2011-07-21,TDG,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2020-12-24,NICE,FRHC,ROL,WST,UTZ
2020-12-28,FRHC,ROL,WST,UTZ,NaN
2020-12-29,NICE,FRHC,WST,UTZ,NaN


In [10]:
testing_score = load_scores(testing_df)
testing_top = get_tops(testing_score.drop(["QQQ", "SPY"], axis=1, level=1, errors='ignore'))
testing_top

C:\Users\sasso\AppData\Local\Temp\ipykernel_28844\4258049907.py:27: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  yearly_ret = (close.pct_change(periods=year) * 100)
C:\Users\sasso\AppData\Local\Temp\ipykernel_28844\4258049907.py:29: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  base_daily_ret = close.pct_change()


Position,1,2,3,4,5
Date,,,,,
2020-01-02,BEP,NaN,NaN,NaN,NaN
2020-01-03,BEP,NaN,NaN,NaN,NaN
2020-02-03,BEP,NaN,NaN,NaN,NaN
2020-02-04,BEP,NaN,NaN,NaN,NaN
2020-02-05,BEP,NaN,NaN,NaN,NaN
...,...,...,...,...,...
2026-02-20,LHX,NaN,NaN,NaN,NaN
2026-02-23,LHX,NaN,NaN,NaN,NaN
2026-02-24,IX,LHX,NaN,NaN,NaN


In [105]:
training_df.loc["2011-01-03"]["Rank"]["AZO"]

np.float64(1.0)

In [90]:
get_scores(training_df.loc["2011-06-30"])

,TotalReturn %,TotalReturn,Volatility,SteadyScore,PositiveDays,PositiveDays %,MaxDrawdown,DistFromSMA50,VolumeSurgeRatio,ATR,FinalScore,Rank
Ticker,,,,,,,,,,,,
OKE,80.349146,88.390805,85.747126,100.0,100.0,61.264822,50.0,50.0,50.0,100.0,100.0,1.0
TDG,80.502760,88.505747,85.172414,50.0,50.0,55.731225,100.0,100.0,100.0,50.0,50.0,2.0


In [11]:
"""
FAIL
"""
top = testing_top.copy()
changes = (top[1].shift(1) != top[1])
exit_date = pd.to_datetime("2011-06-30")
sim = TradingSim(full_df)
avg_stock = []
avg_idx = []
while len(top[1][exit_date:].index) > 0:
    entry_date = top[1][exit_date:].index[0]
    current_stock = top[1].loc[entry_date]
    exit_date = entry_date
    while True:
        exit_date = sim.get_next_trading_day(exit_date + pd.DateOffset(days=80))
        if exit_date not in top.index or top[1].loc[exit_date] != current_stock:
            break
    avg_stock.append(sim.simulate_exit_after(current_stock, entry_date, exit_date) * 100)
    avg_idx.append(sim.simulate_exit_after("QQQ", entry_date, exit_date) * 100)
    print(f"Entring {current_stock}")
    print(f"Entry date {entry_date}")
    print(f"Exit date: {exit_date}")
    print(f"Stock change {avg_stock[-1]}")
    print(f"QQQ change {avg_idx[-1]}")
    print(f"Average Stock Change {sum(avg_stock)/len(avg_stock)}")
    print(f"Average QQQ Change {sum(avg_idx)/len(avg_idx)}")
    print("-----------------------")


Entring BEP
Entry date 2020-01-02 00:00:00
Exit date: 2020-03-23 00:00:00
Stock change -32.033883426610785
QQQ change -20.97370257448089
Average Stock Change -32.033883426610785
Average QQQ Change -20.97370257448089
-----------------------
Entring FNV
Entry date 2020-04-21 00:00:00
Exit date: 2020-07-10 00:00:00
Stock change 14.164410557707063
QQQ change 29.059006970809353
Average Stock Change -8.93473643445186
Average QQQ Change 4.042652198164232
-----------------------
Entring WST
Entry date 2020-07-10 00:00:00
Exit date: 2020-09-28 00:00:00
Stock change 17.363516346146923
QQQ change 5.011935196605667
Average Stock Change -0.16865217425226633
Average QQQ Change 4.365746530978043
-----------------------
Entring LOGI
Entry date 2020-09-28 00:00:00
Exit date: 2020-12-17 00:00:00
Stock change 24.97999234681135
QQQ change 12.193408843481826
Average Stock Change 6.118508956013638
Average QQQ Change 6.322662109103989
-----------------------
Entring AMZN
Entry date 2020-12-17 00:00:00
Exit d

In [62]:
changes.head(20)

Date
2011-06-30     True
2011-07-01    False
2011-07-05     True
2011-07-06    False
2011-07-07    False
2011-07-11    False
2011-07-15    False
2011-07-19    False
2011-07-20    False
2011-07-21    False
2012-01-23     True
2012-01-24    False
2012-01-25    False
2012-01-26    False
2012-01-27    False
2012-01-30    False
2012-01-31    False
2012-02-01    False
2012-02-02    False
2012-02-06    False
Name: 1, dtype: bool

In [ ]:
sim = TradingSim(full_df)

sim.simulate_exit_after("OKE", "2011-06-30", 40) * 100

-9.039305136995198

In [26]:
sim.simulate_exit_after("QQQ", "2011-06-30", 40) * 100


-7.046462606880128

In [84]:
# 1. Set your parameters
streak_length = 3
target_cols = [1, 2, 3]  # The positions we want to track and hold
safety_net_cols = [1, 2, 3, 4, 5]  # Ticker must stay in here to be kept

for col in target_cols:
    hold_name = f'Hold{col}'
    
    # 2. Find streaks for this specific column
    is_streak = pd.Series(True, index=top.index)
    for i in range(1, streak_length):
        is_streak &= (top[col] == top[col].shift(i))
        
    # 3. Create the Hold column: assign ticker if streak met, else NaN
    top[hold_name] = np.where(is_streak, top[col], np.nan)
    
    # 4. Initialize the first row to establish a starting point
    top.iloc[0, top.columns.get_loc(hold_name)] = top.iloc[0][col]
    
    # 5. Forward-fill the empty gaps
    top[hold_name] = top[hold_name].ffill()
    
    # 6. Apply the Safety Net (Highly Optimized Vectorized Check)
    # Check if the currently held ticker is still present ANYWHERE in columns 1 through 5
    is_in_top5 = pd.Series(False, index=top.index)
    for safe_col in safety_net_cols:
        is_in_top5 |= (top[hold_name] == top[safe_col])
        
    # If the held ticker dropped out of the Top 5, force an update to today's ticker in this slot
    top[hold_name] = np.where(is_in_top5, top[hold_name], top[col])
top

Position,1,2,3,4,5,Hold_1,Hold_2,Hold_3,Hold1,Hold2,Hold3
Date,,,,,,,,,,,
2011-01-03,AZO,HRL,LUMN,BRO,NVO,AZO,HRL,LUMN,AZO,HRL,LUMN
2011-01-04,NVO,AZO,LUMN,HRL,TDG,AZO,HRL,LUMN,AZO,HRL,LUMN
2011-01-05,NVO,LUMN,TDG,AZO,HRL,AZO,HRL,LUMN,AZO,HRL,LUMN
2011-01-06,NVO,TDG,LUMN,HRL,AAPL,NVO,HRL,LUMN,NVO,HRL,LUMN
2011-01-07,NVO,TDG,AAPL,HRL,CHRW,NVO,HRL,AAPL,NVO,HRL,AAPL
...,...,...,...,...,...,...,...,...,...,...,...
2020-12-24,BST,TMUS,TSM,QGEN,NICE,BST,TMUS,TSM,BST,TMUS,TSM
2020-12-28,BST,TMUS,QGEN,SONY,AMZN,BST,TMUS,QGEN,BST,TMUS,QGEN
2020-12-29,SONY,QGEN,AMZN,BST,RACE,BST,QGEN,AMZN,BST,QGEN,AMZN


In [84]:
scores_df = get_scores(training_df.loc["2011-06-30"])["Volatility"]
scores_df

Ticker
OKE    85.747126
TDG    85.172414
Name: Volatility, dtype: float64